In [ ]:
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
IMAGE_FOLDER = "/content/drive/MyDrive/poster_images_50k"

In [ ]:
image_files = {
    int(os.path.splitext(file)[0])
    for file in os.listdir(IMAGE_FOLDER)
    if file.lower().endswith(".jpg")
}

print("Available images:", len(image_files))

Available images: 49990


In [ ]:
import os
import pandas as pd

ORIGINAL_DATASET = "/content/drive/MyDrive/movie1M_data.csv"

IMAGE_FOLDER = "/content/drive/MyDrive/poster_images_50k"

OUTPUT_DATASET = "/content/drive/MyDrive/poster_dataset_49990.csv"


In [ ]:


# Original movie dataset
df = pd.read_csv(ORIGINAL_DATASET)

# Available downloaded poster images
image_files = [
    file
    for file in os.listdir(IMAGE_FOLDER)
    if file.lower().endswith(".jpg")
]

# Original row indexes of downloaded images
image_indexes = [
    int(os.path.splitext(file)[0])
    for file in image_files
]

# Keep only the exact rows whose poster image exists
poster_df = df.loc[image_indexes].copy()

# Keep required columns
poster_df = poster_df[
    ["title", "overview", "url", "genre"]
]

# Reset row numbers
poster_df = poster_df.reset_index(drop=True)

# Export clean dataset
poster_df.to_csv(
    OUTPUT_DATASET,
    index=False
)

print("Original Dataset Rows:", len(df))
print("Available Poster Images:", len(image_files))
print("Final Poster Dataset Rows:", len(poster_df))
print("Saved:", OUTPUT_DATASET)

Original Dataset Rows: 1000590
Available Poster Images: 49990
Final Poster Dataset Rows: 49990
Saved: /content/drive/MyDrive/poster_dataset_49990.csv


In [ ]:
IMAGE_FOLDER = "/content/drive/MyDrive/poster_dataset_49990.csv"

files = [
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith(".jpg")
][:50000]

print("Images:", len(files))

NotADirectoryError: [Errno 20] Not a directory: '/content/drive/MyDrive/poster_dataset_49990.csv'

In [ ]:
model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

print("Model loaded")

/tmp/ipykernel_59424/3261971413.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(


Model loaded


In [ ]:
def load_image(file):

    image = Image.open(
        os.path.join(IMAGE_FOLDER, file)
    ).convert("RGB")

    image = image.resize((224, 224))
    image = np.array(image)
    image = preprocess_input(image)

    return image

In [ ]:
BATCH_SIZE = 256
WORKERS = 25

embeddings = []

for i in tqdm(range(0, len(files), BATCH_SIZE)):

    batch_files = files[i:i + BATCH_SIZE]

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        images = list(executor.map(load_image, batch_files))

    images = np.array(images)

    batch_embeddings = model.predict(
        images,
        verbose=0
    )

    embeddings.extend(batch_embeddings)

100%|██████████| 196/196 [18:02<00:00,  5.52s/it]


In [ ]:
embeddings = np.array(embeddings)

np.save(
    "/content/drive/MyDrive/poster_image_embeddings_50k.npy",
    embeddings
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (49990, 1280)


In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

index = faiss.IndexFlatIP(1280)
index.add(embeddings)

faiss.write_index(
    index,
    "/content/drive/MyDrive/poster_index.faiss"
)

print("FAISS index created:", index.ntotal)

FAISS index created: 49990
